# GraphRAG: Uncovering Money Laundering with BigQuery Native Graph and Vector Search

This notebook demonstrates building a GraphRAG solution for Financial Fraud and Anti-Money Laundering (AML) using BigQuery's native Graph capabilities.

## The Scenario: Uncovering a Money Laundering Ring
A bank is investigating a series of suspicious activities. We have several individuals:
*   **Doe**: Owner of a suspected shell company involved in offshore fund routing.
*   **Jacoby**: Under audit for high-volume suspicious transfers from business accounts.
*   **Menville**: A retail customer whose account recently failed KYC verification and has a large loan repayment due.
*   **Smith**: A regular retail customer who occasionally transfers small amounts to friends (the "Innocent Bystander").

**The Mystery:** Menville's account is failing KYC checks. On the surface, it looks like a simple compliance issue. However, the transaction graph reveals a complex money trail: Doe -> Jacoby -> Menville -> Loan.

**The Challenge (The Smith Role):** Menville also received a small transfer of $150 from **Smith**. In a standard investigation, simple association might flag everyone. GraphRAG allows the LLM to see the entire context—the size of the transfers and the audit logs of the senders—to conclude that while Smith is part of the graph, he is not part of the crime.

**The GraphRAG Solution:**
1.  **Vector Search** identifies Menville and his associated audit flags.
2.  **Graph Traversal** discovers the hidden money trail (Doe -> Jacoby -> Menville) AND the noise (Smith -> Menville).
3.  **LLM Reasoning** synthesizes the amounts and logs to isolate the money laundering ring while correctly identifying Smith as an irrelevant connection.

In [ ]:
# Configuration
GCP_PROJECT_ID = "[GCP PROJECT ID]"
REGION = "us-central1"
BQ_DATASET_ID = "fingraph_rag"
MODEL_NAME = "gemini-2.5-flash"
EMBEDDING_MODEL_NAME = "text-embedding-005"


### 1. Setup and Initialization
We initialize the BigQuery and Vertex AI clients.


In [ ]:
!pip install --quiet langchain-google-vertexai langchain-core google-cloud-bigquery vertexai


In [ ]:
import vertexai
from google.cloud import bigquery

bq_client = bigquery.Client(project=GCP_PROJECT_ID)
vertexai.init(project=GCP_PROJECT_ID, location=REGION)


### 2. Create Tables and Schema
We define the schema for our Financial Graph.


In [ ]:
!bq mk --location=US --dataset {BQ_DATASET_ID}


In [ ]:
%%bigquery

CREATE TABLE IF NOT EXISTS `fingraph_rag.Account` (id INT64, create_time TIMESTAMP, is_blocked BOOL, type STRING);
CREATE TABLE IF NOT EXISTS `fingraph_rag.Loan` (id INT64, loan_amount FLOAT64, balance FLOAT64, create_time TIMESTAMP, interest_rate FLOAT64);
CREATE TABLE IF NOT EXISTS `fingraph_rag.Person` (id INT64, name STRING);
CREATE TABLE IF NOT EXISTS `fingraph_rag.AccountRepayLoan` (id INT64, loan_id INT64, amount FLOAT64, create_time TIMESTAMP);
CREATE TABLE IF NOT EXISTS `fingraph_rag.AccountTransferAccount` (id INT64, to_id INT64, amount FLOAT64, create_time TIMESTAMP);
CREATE TABLE IF NOT EXISTS `fingraph_rag.PersonOwnAccount` (id INT64, account_id INT64, create_time TIMESTAMP);
CREATE TABLE IF NOT EXISTS `fingraph_rag.AccountAudits` (id INT64, audit_timestamp TIMESTAMP, audit_details STRING, embedding ARRAY<FLOAT64>);


### 3. Insert the Compelling Dataset
We insert the entities and their relationships to form our money trail.


In [ ]:
%%bigquery

INSERT INTO `fingraph_rag.Account` VALUES (10,'2020-01-10 06:22:20.222',false,'brokerage account'), (20,'2020-01-27 17:55:09.206',false,'checking account'), (30,'2020-02-15 09:12:33.111',false,'savings account'), (40,'2019-11-05 14:33:10.000',false,'business account');
INSERT INTO `fingraph_rag.Loan` VALUES (100,2022278.5,123359.0,'2020-03-18 16:42:57.719',0.064), (200,50000.0,45000.0,'2020-03-23 19:03:05.567',0.097), (300, 15000.0, 10000.0, '2020-05-10 10:00:00.000', 0.05);
INSERT INTO `fingraph_rag.Person` VALUES (1,'Jacoby'), (2,'Menville'), (3,'Smith'), (4,'Doe');
INSERT INTO `fingraph_rag.AccountTransferAccount` VALUES (40,10,25000.0,'2020-08-01 10:00:00.000'), (10,20,24000.0,'2020-08-29 15:28:58.647'), (30,20,150.0,'2020-09-01 12:00:00.000');
INSERT INTO `fingraph_rag.AccountRepayLoan` VALUES (10,100,56809.8,'2020-12-12 07:25:02.597'), (20,200,20000.0,'2021-01-18 01:40:25.317');
INSERT INTO `fingraph_rag.PersonOwnAccount` VALUES (1,10,'2020-01-10 06:22:20.222'), (2,20,'2020-01-27 17:55:09.206'), (3,30,'2020-02-15 09:12:33.111'), (4,40,'2019-11-05 14:33:10.000');
INSERT INTO `fingraph_rag.AccountAudits` (id, audit_timestamp, audit_details) VALUES (10, '2020-05-14 06:57:02', 'Account 10 (Jacoby) flagged by AML system for suspicious high-volume transfers from offshore business accounts.'), (20, '2021-03-09 02:51:45', 'Account 20 (Menville) failed KYC verification. Linked source of funds is unverified and customer is unresponsive.'), (40, '2020-07-20 09:00:00', 'Account 40 (Doe) under investigation as a suspected shell company involved in illicit activities.');


### 4. Create BigQuery Property Graph
We define the `FinGraph` using BigQuery's native Graph DDL.


In [ ]:
%%bigquery
CREATE OR REPLACE PROPERTY GRAPH `fingraph_rag.FinGraph`
 NODE TABLES (
   `fingraph_rag.Account` KEY (id) LABEL Account PROPERTIES (id, type, is_blocked),
   `fingraph_rag.Loan` KEY (id) LABEL Loan PROPERTIES (id, loan_amount, balance),
   `fingraph_rag.Person` KEY (id) LABEL Person PROPERTIES (id, name)
 )
 EDGE TABLES(
   `fingraph_rag.AccountRepayLoan`
     KEY (id, loan_id, create_time)
     SOURCE KEY (id) REFERENCES `fingraph_rag.Account` (id)
     DESTINATION KEY (loan_id) REFERENCES `fingraph_rag.Loan` (id)
     LABEL Repays PROPERTIES (amount, create_time),
   `fingraph_rag.AccountTransferAccount`
     KEY (id, to_id, create_time)
     SOURCE KEY (id) REFERENCES `fingraph_rag.Account` (id)
     DESTINATION KEY (to_id) REFERENCES `fingraph_rag.Account` (id)
     LABEL Transfers PROPERTIES (amount, create_time),
   `fingraph_rag.PersonOwnAccount`
     KEY (id, account_id)
     SOURCE KEY (id) REFERENCES `fingraph_rag.Person` (id)
     DESTINATION KEY (account_id) REFERENCES `fingraph_rag.Account` (id)
     LABEL Owns PROPERTIES (create_time)
 );


### 4.1 Visualize the Entire Dataset as a Graph
If we want to see every entity and every transaction in our dataset all at once, we can use a generic GQL pattern match `(src)-[e]->(dst)` to capture all edges and their connecting nodes.\n\nRun this cell with the `--graph` flag in a compatible environment (like BigQuery Studio) to visualize the complete FinGraph network.

In [ ]:
%%bigquery --graph
GRAPH `fingraph_rag.FinGraph`
MATCH (src)-[e]->(dst)
RETURN TO_JSON([
  TO_JSON(src),
  TO_JSON(e),
  TO_JSON(dst)
  ]) AS result;

### 5. Generate Embeddings for Audit Logs
We generate embeddings for the semantic search part of the RAG pipeline.


In [ ]:
%%bigquery
UPDATE `fingraph_rag.AccountAudits` SET embedding = AI.EMBED(audit_details, endpoint => 'text-embedding-005').result
WHERE ARRAY_LENGTH(embedding) = 0;


### 6. Define the GraphRAG Pipeline
We create a custom LangChain retriever that combines Vector Search and Native Graph MATCH queries.


In [ ]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from typing import List

class FinGraphRetriever(BaseRetriever):
    project: str
    dataset: str

    def _get_relevant_documents(self, query: str) -> List[Document]:
        # 1. Vector Search
        vector_query = f"""
            SELECT id, audit_details
            FROM `{self.dataset}.AccountAudits`
            ORDER BY COSINE_DISTANCE(embedding, AI.EMBED(@query, endpoint => 'text-embedding-005').result)
            LIMIT 1
        """
        res = bq_client.query(vector_query, job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("query", "STRING", query)]
        )).result()

        start_id = None
        audit_text = ""
        for row in res:
            start_id = row.id
            audit_text = row.audit_details

        if not start_id: return []

        # 2. Native Graph Traversal
        graph_query = f"""
            GRAPH `{self.dataset}.FinGraph`
            MATCH
              (sender_person:Person)-[:Owns]->(sender_acc:Account)
              -[tx:Transfers]->
              (a:Account {{id: @id}})
              -[repays:Repays]->(l:Loan),
              (owner:Person)-[:Owns]->(a)
            RETURN
              owner.name as owner_name,
              a.type as account_type,
              sender_person.name as sender_name,
              tx.amount as transfer_amount,
              repays.amount as repayment_amount,
              l.id as loan_id
        """
        graph_res = bq_client.query(graph_query, job_config=bigquery.QueryJobConfig(
            query_parameters=[bigquery.ScalarQueryParameter("id", "INT64", start_id)]
        )).result()

        context_docs = [Document(page_content=f"Primary Audit Log (Target Account): {audit_text}")]
        sender_names = []
        for row in graph_res:
            sender_names.append(row['sender_name'])
            doc_str = (f"Account Owner: {row['owner_name']} (Account Type: {row['account_type']}). "
                       f"Received transfer of ${row['transfer_amount']} from {row['sender_name']}. "
                       f"Made loan repayment of ${row['repayment_amount']} to Loan {row['loan_id']}.")
            context_docs.append(Document(page_content=doc_str))

        if sender_names:
            names_list = "','".join(sender_names)
            sender_audit_query = f"""
                SELECT p.name, au.audit_details
                FROM `{self.dataset}.AccountAudits` au
                JOIN `{self.dataset}.Account` a ON au.id = a.id
                JOIN `{self.dataset}.PersonOwnAccount` poa ON a.id = poa.account_id
                JOIN `{self.dataset}.Person` p ON poa.id = p.id
                WHERE p.name IN ('{names_list}')
            """
            sender_audits = bq_client.query(sender_audit_query).result()
            for row in sender_audits:
                context_docs.append(Document(page_content=f"Audit Log for Sender {row['name']}: {row['audit_details']}"))

        return context_docs


### 7. Run the Fraud Investigation
Finally, we run the GraphRAG pipeline to generate a detailed fraud report.


In [ ]:
from langchain_google_vertexai import ChatVertexAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatVertexAI(model_name=MODEL_NAME)
retriever = FinGraphRetriever(project=GCP_PROJECT_ID, dataset=BQ_DATASET_ID)

prompt = ChatPromptTemplate.from_template("""
You are a Lead Fraud Analyst. Use the following audit logs and graph transaction history to answer the question.
Your goal is to connect the dots between the entities and explain the flow of funds.
If you see transfers from flagged users or shell companies, highlight the money laundering risk.

Context: {context}

Question: {question}

Detailed Fraud Report:
""")

chain = (
    {"context": retriever , "question": lambda x: x}
    | prompt
    | llm
    | StrOutputParser()
)

question = "Why is Menville's loan repayment at risk? Flag any suspicious activity if you notice."
print(chain.invoke(question))


### 8. Visualize the Money Laundering Trail
We can use a GQL query to trace the entire path from the suspicious shell company owner (**Doe**) through the intermediary (**Jacoby**) to the final target (**Menville**) and the **Loan** repayment.

Running this with the `--graph` flag in BigQuery Studio (or similar visualization tools) will render the nodes and edges as a connected network.

In [ ]:
%%bigquery --graph
    GRAPH `fingraph_rag.FinGraph`
     MATCH
       (p_shell:Person)-[o1:Owns]->(acc_shell:Account)-[t1:Transfers]->(acc_fraud:Account)-[t2:Transfers]->(acc_target:Account)-[r:Repays]->(l:Loan),
       (p_fraud:Person)-[o2:Owns]->(acc_fraud),
       (p_target:Person)-[o3:Owns]->(acc_target)
     WHERE p_target.name = 'Menville' AND p_fraud.name = 'Jacoby' AND p_shell.name = 'Doe'
     RETURN TO_JSON([
      TO_JSON(p_shell), TO_JSON(o1), TO_JSON(acc_shell),
      TO_JSON(t1), TO_JSON(acc_fraud), TO_JSON(p_fraud), TO_JSON(o2),
      TO_JSON(t2), TO_JSON(acc_target), TO_JSON(p_target), TO_JSON(o3),
      TO_JSON(r), TO_JSON(l)
    ]) AS result;



### 9. Cleanup
If you want to delete the dataset and all associated tables and graphs to avoid ongoing storage costs, run the following cell.

In [ ]:
# Delete the BigQuery dataset and all its contents
!bq rm -r -f {BQ_DATASET_ID}